## 필요 라이브러리 호출

In [2]:
!pip install wordcloud

In [1]:
!pip install JPype1

In [1]:
!pip install pyahocorasick

In [1]:
import pandas as pd
import numpy as np
import re #Regular Expression 특수문자, 반복 패턴 처리 용이
from collections import Counter
import os
import sys

#시각화
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
font_path = 'C:/Windows/Fonts/malgun.ttf' 

wc = WordCloud(
    font_path=font_path,
    background_color='white',
    width=800,
    height=600
)

import plotly.graph_objects as go  # Sankey Diagram

from konlpy.tag import Okt
'''
okt = Okt()
nouns = okt.nouns("맛있는 호박고구마 5kg 특가")
print(nouns) # 출력: ['호박고구마', '특가']
'''

plt.rcParams['font.family'] = ['Malgun Gothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

## 변수 설명

product_name: 1차 정제 제품명

category_large: 대분류 카테고리

category_medium: 중분류 카테고리

item_type: 제품의 카테고리 태그

grocery_item_name: 식재료 이름(타겟)

brand_name: 제품의 브랜드 명

## 데이터 전처리

In [2]:
grocery_df = pd.read_csv('C:/Users/LG/OneDrive/Desktop/REF_Classification_For_Ingredient_Recognition/ML_dataset_smapled/refined_grocery_dataset_sampled/ML_grocery_data_sampled.csv')

print("Grocery data shape:", grocery_df.shape)

display(grocery_df.head(5))

grocery_df.info()

Grocery data shape: (1143, 6)


,product_name,category_large,category_medium,item_type,grocery_item_name,brand_name
0,[마이너피겨스] 유기농 오트음료,음료,주스,BEVERAGE,오트음료,마이너피겨스
1,[마이셰프] 찹스테이크,간편식,밀키트,RTC_MEAL,밀키트,마이셰프
2,[건강한우리집비옴] 생 아몬드 분말,조미료,가루분말,SAUCE_SEASONING,아몬드 분말,건강한우리집비옴
3,[김구원선생] 매일 마시는 국산 콩물,채소/곡물류,콩/두부,SIMPLE_PROCESSED,콩물,김구원선생
4,스코티시 리더 슈프림,술,기타 주류,ALCOHOL,위스키,스코티시


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_name       1143 non-null   object
 1   category_large     1143 non-null   object
 2   category_medium    1143 non-null   object
 3   item_type          1143 non-null   object
 4   grocery_item_name  1143 non-null   object
 5   brand_name         1143 non-null   object
dtypes: object(6)
memory usage: 53.7+ KB


### 제품명 정제 및 컬럼 생성 메서드

In [8]:
from datetime import datetime
import json
# 파서 import
from product_name_parser import REFProductNameParser, create_brand_matcher

class REF_Grocery_Data_Cleaner_prototype:
        self.new_brands_added = set()
        self.update_parser()

    def update_parser(self):
        # 브랜드 매처 생성 로직을 사용 -> 브랜드 검색 사전 만들기
        self.matcher = create_brand_matcher(self.brand_dictionary)
        # 파서 클래스를 인스턴스
        self.parser = REFProductNameParser(brand_matcher=self.matcher)

    def refine_product_name(self, product_name: str):
        #
        if not product_name or pd.isna(product_name):
            return product_name
        
        # 파서로 분석 수행 (브랜드 추출, 노이즈 제거 등)
        result = self.parser.parse(product_name)
        
        # 새로운 브랜드 발견 업데이트
        found_brand = result.brand_name
        if found_brand and found_brand not in self.brand_dictionary:
            self.brand_dictionary.add(found_brand)
            self.new_brands_added.add(found_brand)
            # 다음 행 처리에 즉시 반영
            self.update_parser()

        return result.refined_text

    def add_refined_column(self, df, input_col='product_name'):
        # cleaned_product_name 이라는 컬럼 생성
        print(f"[{input_col}] 컬럼 정제 및 브랜드 수집 중...")
        df['refined_product_name'] = df[input_col].apply(self.refine_product_name)
        return df

    def save_updated_dictionary(self, target_path):
        # 기존 브랜드 사전과 비교해 새로운 브랜드가 있을시 기존 브랜드 사전에 추가 및 데이터 저장
        if not self.new_brands_added:
            print("-" * 50)
            print("새로운 브랜드가 발견되지 않아 저장을 건너뜁니다.")
            print("-" * 50)
            return

        # 날짜 포맷팅
        date_str = datetime.now().strftime("%y_%m_%d")
        file_name = f"{date_str}_predict_grocery_dictionary.json"
        full_path = os.path.join(target_path, file_name)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)

        # 브랜드 사전을 정렬된 리스트(배열)로 변환하여 JSON 저장
        brand_list = sorted(list(self.brand_dictionary))
        with open(full_path, 'w', encoding='utf-8') as f:
            json.dump(brand_list, f, ensure_ascii=False, indent=4)
        
        print("-" * 50)
        print(f"신규 브랜드 {len(self.new_brands_added)}개 추가 발견.")
        print(f"JSON 사전 저장 완료: {full_path}")
        print("-" * 50)

# brand_name 컬럼의 유효한 값들을 중복 없애서 사전으로 만들기.
brand_dictionary = set(grocery_df['brand_name'].dropna().unique())

# 클리너 인스턴스 생성
cleaner = REF_Grocery_Data_Cleaner_prototype(brand_dictionary)

# 데이터 정제 및 새로운 컬럼 추가
# grocery_df에 직접 적용.
grocery_df = cleaner.add_refined_column(grocery_df, input_col='product_name')

# 결과 확인 (상위 5개 행)
print(grocery_df[['product_name', 'refined_product_name']].head())

# 지정된 경로에 JSON 사전 저장
TARGET_DIR = r"C:\Users\LG\OneDrive\Desktop\REF_Classification_For_Ingredient_Recognition\ML_dataset_smapled\grocery_classification_dataset\predict_brand_name" 
cleaner.save_updated_dictionary(TARGET_DIR)

[product_name] 컬럼 정제 및 브랜드 수집 중...
           product_name refined_product_name
0     [마이너피겨스] 유기농 오트음료             유기농 오트음료
1          [마이셰프] 찹스테이크                찹스테이크
2   [건강한우리집비옴] 생 아몬드 분말             생 아몬드 분말
3  [김구원선생] 매일 마시는 국산 콩물         매일 마시는 국산 콩물
4           스코티시 리더 슈프림               리더 슈프림
--------------------------------------------------
새로운 브랜드가 발견되지 않아 저장을 건너뜁니다.
--------------------------------------------------


술 제품들의 제품명이 잘림 -> 브랜드를 없애고 정제 제품명으로 바꾼 것이 문제.

In [7]:
from datetime import datetime
import json
# 파서 import
from product_name_parser import REFProductNameParser, create_brand_matcher

class REF_Grocery_Data_Cleaner:
    def __init__(self, brand_set: set):
        self.brand_dictionary = brand_set
        self.new_brands_added = set()
        self.update_parser()

    def update_parser(self):
        # 브랜드 매처 및 파서 초기화
        self.matcher = create_brand_matcher(self.brand_dictionary)
        self.parser = REFProductNameParser(brand_matcher=self.matcher)

    def refine_product_name(self, row, input_col='product_name', category_col='category_large'):
        product_name = row[input_col]
        category = str(row[category_col]).strip() if category_col in row else ""

        if not product_name or pd.isna(product_name):
            return product_name
        
        # 파서 실행 (브랜드 추출 및 괄호/특수문자 등 노이즈 제거)
        result = self.parser.parse(product_name)
        
        # 브랜드 사전 업데이트 (신규 브랜드 발견 시)
        found_brand = result.brand_name
        if found_brand and found_brand not in self.brand_dictionary:
            self.brand_dictionary.add(found_brand)
            self.new_brands_added.add(found_brand)
            self.update_parser()

        # 술 카테고리 여부에 따른 분기 처리
        if category == '술':
            # 브랜드가 식별되었다면, 정제된 텍스트 앞에 브랜드를 다시 붙여줌
            # 파서가 정제 과정에서 브랜드를 삭제하더라도 여기서 복구됨
            refined = result.refined_text
            if found_brand and found_brand not in refined:
                return f"{found_brand} {refined}".strip()
            return refined
        else:
            # 술이 아닌 경우 기존 로직대로 정제된 텍스트(일반적으로 브랜드 제외) 반환
            return result.refined_text

    def add_refined_column(self, df, input_col='product_name', category_col='category_large'):
        print(f"[{input_col}] 정제 작업 시작 (대상: {len(df)}행)...")
        
        # 행(row) 전체를 전달하여 카테고리 값을 참조
        df['refined_product_name'] = df.apply(
            lambda row: self.refine_product_name(row, input_col, category_col), 
            axis=1
        )
        return df

    def save_updated_dictionary(self, target_path):
        if not self.new_brands_added:
            print("-" * 50)
            print("새로운 브랜드가 발견되지 않아 저장을 건너뜁니다.")
            print("-" * 50)
            return

        date_str = datetime.now().strftime("%y_%m_%d")
        file_name = f"{date_str}_predict_grocery_dictionary.json"
        full_path = os.path.join(target_path, file_name)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)

        brand_list = sorted(list(self.brand_dictionary))
        with open(full_path, 'w', encoding='utf-8') as f:
            json.dump(brand_list, f, ensure_ascii=False, indent=4)
        
        print("-" * 50)
        print(f"작업 완료: 신규 브랜드 {len(self.new_brands_added)}개 추가됨.")
        print(f"저장 경로: {full_path}")
        print("-" * 50)

In [8]:
brand_dictionary = set(grocery_df['brand_name'].dropna().unique())
ref_cleaner = REF_Grocery_Data_Cleaner(brand_dictionary)

# 3. 데이터 정제 (대분류 컬럼 'category_large' 참조)
grocery_df = ref_cleaner.add_refined_column(grocery_df, input_col='product_name', category_col='category_large')

# 4. 결과 확인
print(grocery_df[['category_large', 'product_name', 'refined_product_name']].head())

# 5. 지정된 경로에 JSON 사전 저장
TARGET_DIR = r"C:\Users\LG\OneDrive\Desktop\REF_Classification_For_Ingredient_Recognition\ML_dataset_smapled\grocery_classification_dataset\predict_brand_name"
ref_cleaner.save_updated_dictionary(TARGET_DIR)

[product_name] 정제 작업 시작 (대상: 1143행)...
  category_large          product_name refined_product_name
0             음료     [마이너피겨스] 유기농 오트음료             유기농 오트음료
1            간편식          [마이셰프] 찹스테이크                찹스테이크
2            조미료   [건강한우리집비옴] 생 아몬드 분말             생 아몬드 분말
3         채소/곡물류  [김구원선생] 매일 마시는 국산 콩물         매일 마시는 국산 콩물
4              술           스코티시 리더 슈프림          스코티시 리더 슈프림
--------------------------------------------------
새로운 브랜드가 발견되지 않아 저장을 건너뜁니다.
--------------------------------------------------


In [4]:
# 형태소 분석기 및 불용어 설정
okt = Okt()

# 식재료와 상관없는 빈출 불용어 리스트
stop_words = ['기획', '특가', '증정', '박스', '세트', '국내산', '내외', '중량', '입', '행사', '무료', '배송', '[26년 설]',
             ]

# 문장에서 명사만 추출하고 불용어를 제거하는 함수
def get_nouns_fast(text):
    if pd.isna(text): return ""
    # 명사 추출
    nouns = okt.nouns(str(text))
    # 한 글자 단어 제거 및 불용어 제거
    return [n for n in nouns if len(n) > 1 and n not in stop_words]

# 원본 제품명에서 명사 추출
grocery_df['nouns_original'] = grocery_df['product_name'].apply(get_nouns_fast)
# 정제된 제품명에서 명사 추출
grocery_df['nouns_cleaned'] = grocery_df['cleaned_product_name'].apply(get_nouns_fast)

# 리스트를 풀어서 전체 빈도 계산
all_nouns_product_name = [word for nouns in df['nouns_original'] for word in nouns]
all_nouns_cleaned_product_name = [word for nouns in df['nouns_cleaned'] for word in nouns]

counts_product_name = Counter(all_nouns_product_name)
counts_cleaned_product_name = Counter(all_nouns_cleaned_product_name)

# 워드 클라우드 생성
def draw_wordcloud(word_counts, title, ax):
    wc = WordCloud(
        font_path=font_path,
        background_color='white',
        width=800,
        height=600,
        max_words=100
    ).generate_from_frequencies(word_counts)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=20)
    ax.axis('off')

# 4. 결과 출력
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# 왼쪽: 원본 데이터 워드클라우드
draw_wordcloud(counts_product_name, "[Before] Original Product Name", axes[0])

# 오른쪽: 브랜드 제거 후 워드클라우드
draw_wordcloud(counts_cleaned_product_name, "[After] Cleaned Product Name", axes[1])

plt.tight_layout()
plt.show()

# 5. 상위 단어 수치 비교 출력
print("\n[비교 결과 상위 10개 단어]")
print(f"{'원본(product_name)':<15} | {'정제후(cleaned_product_name)':<15}")
print("-" * 40)
for (w1, c1), (w2, c2) in zip(counts_product_name.most_common(10), counts_cleaned_product_name.most_common(10)):
    print(f"{w1}({c1}):<15 | {w2}({c2}):<15")

JVMNotFoundException: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.